# 01. Quickstart

Минимальный пример: обучить маленькую модель на готовых спектрограммах
и проверить, что весь пайплайн работает.

**Что здесь есть:**
- загрузка данных через `DataPipelineDone`
- модель `SmallAudioCNN` (быстрая, для smoke-теста)
- обёртка `LitAudioModel`
- `Trainer` без MLflow (только прогресс-бар)
- `trainer.test()` на лучшем чекпоинте

**Чего здесь нет:**
- MLflow

Если этот ноутбук отработал без ошибок — окружение готово,
можно переходить к `02_experiment_raw.ipynb`.

In [7]:

import sys
from pathlib import Path
import os
import warnings
warnings.filterwarnings("ignore", ".*does not have many workers.*")

# Чтобы `import src` работал из notebooks/ — добавляем корень проекта в sys.path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

import torch
import lightning as L

from src.constants import C
from src.data import DataPipelineDone
from src.models import SmallAudioCNN, LitAudioModel
from src.training import make_trainer

print(f"Python:     {sys.version.split()[0]}")
print(f"PyTorch:    {torch.__version__}")
print(f"Lightning:  {L.__version__}")
print(f"CUDA:       {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:        {torch.cuda.get_device_name(0)}")

Python:     3.13.5
PyTorch:    2.9.1+cu126
Lightning:  2.6.6
CUDA:       True
GPU:        NVIDIA GeForce RTX 3070 Laptop GPU


## Данные

Используем `DataPipelineDone` -- готовые спектрограммы из `.npz`.
Путь к файлу поменяйте на свой.

`DataPipelineDone` -- это `LightningDataModule`: он сам создаст
train/val/test датасеты и DataLoader'ы, когда `Trainer` вызовет `setup()`.

In [2]:
DATA_PATH = PROJECT_ROOT / "data" / "done" / "base_numpy_dataset.npz"

dm = DataPipelineDone(
    data_path=str(DATA_PATH),
    test_size=C.TEST_SIZE,
    val_size=C.VAL_SIZE,
    seed=C.SEED,
    batch_size=C.BATCH_SIZE,
    mode="Conv",
    augment_train=True,
)

# Вызываем setup() вручную, чтобы:
#   1) увидеть размеры сплитов до обучения
#   2) получить dm.num_categories / dm.num_targets для модели
dm.setup()

print(f"train: {len(dm.train_dataset)} | "
      f"val: {len(dm.val_dataset)} | "
      f"test: {len(dm.test_dataset)}")
print(f"num_categories: {dm.num_categories} | num_targets: {dm.num_targets}")

Загрузка данных из d:\Works\vs_code\ComputerVision\ProjectListen\data\done\base_numpy_dataset.npz...
  Спектрограммы: (6281, 256, 173), dtype=float32
  Категории:     (6281, 5), dtype=float32
  Таргеты:       (6281, 7), dtype=float32
Создание датасетов (done):
  - test: 943
  - train: 4537
  - val:   801
train: 4537 | val: 801 | test: 943
num_categories: 5 | num_targets: 7


In [3]:
# Один батч — убеждаемся, что формы и типы корректны
x, y_cat, y_tgt = next(iter(dm.train_dataloader()))
print(f"x:     {tuple(x.shape)}  dtype={x.dtype}")
print(f"y_cat: {tuple(y_cat.shape)} dtype={y_cat.dtype}")
print(f"y_tgt: {tuple(y_tgt.shape)} dtype={y_tgt.dtype}")
assert x.dim() == 4, "Ожидается (B, C, H, W)"
assert y_cat.dim() == 2 and y_cat.shape[1] == dm.num_categories
assert y_tgt.dim() == 2 and y_tgt.shape[1] == dm.num_targets

x:     (32, 1, 256, 173)  dtype=torch.float32
y_cat: (32, 5) dtype=torch.float32
y_tgt: (32, 7) dtype=torch.float32


## Модель

`SmallAudioCNN` — три свёрточных блока, обучается за секунды.
Нужна, чтобы проверить пайплайн, а не чтобы получить хорошее качество.

Свою модель пишите здесь же: любой `nn.Module`, возвращающий
`(logits_cat, logits_tgt)`.

In [4]:
base_model = SmallAudioCNN(
    in_channels=1,
    num_categories=dm.num_categories,
    num_targets=dm.num_targets,
    base_channels=16,
)

lit_model = LitAudioModel(
    model=base_model,
    num_categories=dm.num_categories,
    num_targets=dm.num_targets,
    lr=1e-3,
    weight_decay=1e-4,
)

n_params = sum(p.numel() for p in base_model.parameters())
print(f"Параметров в base_model: {n_params:,}")

Параметров в base_model: 24,188


## Trainer

`make_trainer` собирает `Trainer` с callbacks (`ModelCheckpoint`,
`EarlyStopping`, `LearningRateMonitor`). MLflow здесь **не** включаем --
для quickstart достаточно прогресс-бара.

`monitor="val_f1_mean"` -- метрика, по которой сохраняется лучший
чекпоинт и срабатывает early stopping.

In [5]:
trainer, _ = make_trainer(
    experiment_name="quickstart",
    run_name="smoke_test",
    max_epochs=5,
    monitor="val_f1_mean",
    mode="max",
    patience=3,
    checkpoint_dir=str(PROJECT_ROOT / "checkpoints" / "quickstart"),
    enable_progress_bar=True,
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


## Обучение

5 эпох на `SmallAudioCNN` — меньше минуты на GPU, пара минут на CPU.
Если val_f1_mean не растёт — это нормально для smoke-теста.

In [8]:
trainer.fit(lit_model, datamodule=dm)
print(f"\nBest checkpoint: {trainer.checkpoint_callback.best_model_path}")
print(f"Best val_f1_mean: {trainer.checkpoint_callback.best_model_score:.4f}")

d:\Programs\Anaconda\Lib\site-packages\lightning\pytorch\callbacks\model_checkpoint.py:881: Checkpoint directory D:\Works\vs_code\ComputerVision\ProjectListen\checkpoints\quickstart exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name          ┃ Type               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ model         │ SmallAudioCNN      │ 24.2 K │ train │     0 │
│ 1  │ train_acc_cat │ MulticlassAccuracy │      0 │ train │     0 │
│ 2  │ train_acc_tgt │ MulticlassAccuracy │      0 │ train │     0 │
│ 3  │ val_acc_cat   │ MulticlassAccuracy │      0 │ train │     0 │
│ 4  │ val_acc_tgt   │ MulticlassAccuracy │      0 │ train │     0 │
│ 5  │ val_f1_cat    │ MulticlassF1Score  │      0 │ train │     0 │
│ 6  │ val_f1_tgt    │ MulticlassF1Score  │      0 │ train │     0 │
│ 7  │ test_acc_cat  │ MulticlassAccuracy │      0 │ train │     0 │
│ 8  │ test_acc_tgt  │ MulticlassAccuracy │      0 │ train │     0 │
│ 9  │ test_f1_cat   │ MulticlassF1Score  │      0 │ train │     0 │
│ 10 │ test_f1_tgt   │ MulticlassF1Score  │      0 │ train │     0 │
└────┴───────────────┴────────────────────┴────────┴───────┴───────┘

Trainable params: 24.2 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 24.2 K                                                                                               
Total estimated model params size (MB): 0.097                                                                      
Modules in train mode: 27                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=5` reached.



Best checkpoint: D:\Works\vs_code\ComputerVision\ProjectListen\checkpoints\quickstart\best-epoch=04-val_f1_mean=0.712.ckpt
Best val_f1_mean: 0.7121


## Тест на лучшем чекпоинте

`ckpt_path="best"` -- Lightning загрузит лучший чекпоинт, сохранённый
`ModelCheckpoint` во время `fit()`.

In [10]:
test_results = trainer.test(lit_model, datamodule=dm, ckpt_path="best", verbose=False)

Restoring states from the checkpoint path at D:\Works\vs_code\ComputerVision\ProjectListen\checkpoints\quickstart\best-epoch=04-val_f1_mean=0.712.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at D:\Works\vs_code\ComputerVision\ProjectListen\checkpoints\quickstart\best-epoch=04-val_f1_mean=0.712.ckpt


Output()

MisconfigurationException: Could not find the `LightningModule` attribute for the `torchmetrics.Metric` logged. You can fix this by calling `self.log(test_f1_mean, ..., metric_attribute=name)` where `name` is one of ['train_acc_cat', 'train_acc_tgt', 'val_acc_cat', 'val_acc_tgt', 'val_f1_cat', 'val_f1_tgt', 'test_acc_cat', 'test_acc_tgt', 'test_f1_cat', 'test_f1_tgt']